***1️⃣ Estructura física del proceso***
El pasteurizador funciona así:

leche fría
     ↓
intercambiador regenerativo
     ↓
calentamiento con agua caliente
     ↓
tubo de retención (pasteurización)
     ↓
enfriamiento con agua fría


Las utilidades principales son:

vapor → calienta el agua de servicio
agua → enfría la leche después

_____

1️⃣ temp_ambiente: Temperatura ambiente de la sala de proceso.

**Afecta a:**
- eficiencia de refrigeración
- pérdidas térmicas
- temperatura de materias primas
- consumo energético global

**Cómo se mide:**
- Sensor ambiental industrial

**De qué depende:**
- estación del año
- ventilación de planta
- carga térmica de equipos

**Valores realistas:**
15°C – 30°C

2️⃣ temp_entrada_leche: Temperatura de la leche cruda al entrar al pasteurizador.

**Afecta a:**
- tanque refrigerado
- intercambiador regenerativo

**Cómo se mide:**
- Sensor PT100 en tubería de entrada.

**De qué depende:**
- refrigeración del tanque
- temperatura ambiente
- tiempo de almacenamiento

temp_entrada = temp_tanque + pérdidas térmicas

**Valores realistas:**
3°C – 6°C

3️⃣ flujo_leche_lh: Caudal volumétrico de leche que atraviesa el pasteurizador.

**Cómo se mide:**
- Caudalímetro

**De qué depende:**
- tamaño de planta
- producción deseada
- bomba de proceso

temp_entrada = temp_tanque + pérdidas térmicas

**Valores realistas:**
2500 – 5000 L/h

4️⃣ temp_setpoint_leche: Temperatura objetivo fijada por el controlador. Es la temperatura que el PLC intenta mantener.

**Cómo se define:**
La fija el operador o la receta de proceso.

**De qué depende:**
- tamaño de planta
- producción deseada
- bomba de proceso

temp_entrada = temp_tanque + pérdidas térmicas

**Valores realistas:**
Pasteurización HTST estándar: 72°C
Pero muchas plantas operan más alto para margen: 72 – 78°C
Más alto empieza a afectar calidad.

5️⃣ temp_proceso_leche: Temperatura real medida en el tubo de retención. Esto es lo que realmente determina la pasteurización.

**Diferencia con setpoint:**
temp_real = setpoint + error_control
donde el error sale de:
- dinámica térmica
- PID
- rudio del sensor

**Desviación realista en industria:**
±0.1 – ±0.5°C

**Valores realistas:**
Pasteurización HTST estándar: 72°C
Pero muchas plantas operan más alto para margen: 72 – 78°C
Más alto empieza a afectar calidad.

6️⃣ temp_agua_servicio: Temperatura del agua caliente usada para calentar la leche.

**Proviene de:**

- circuito de agua caliente
- vapor + intercambiador

El agua debe estar más caliente que la leche.
**Diferencia típica:**
ΔT = 5 – 15°C

**Desviación realista en industria:**
±0.1 – ±0.5°C

**Valores realistas:**
Pasteurización HTST estándar: 72°C
Pero muchas plantas operan más alto para margen: 72 – 78°C
Más alto empieza a afectar calidad.

7️⃣ horas_desde_limpieza: Tiempo desde la última limpieza CIP.

**Qué indica:**
- proteínas
- minerales
- biofilm

Esto se llama **fouling.**

**Valores realistas:**
Plantas limpian cada 8 – 20 h

**Valores realistas:**
Pasteurización HTST estándar: 72°C
Pero muchas plantas operan más alto para margen: 72 – 78°C
Más alto empieza a afectar calidad.

8️⃣ presion_diferencial_bar: Diferencia de presión entre entrada y salida del intercambiador.

**Qué indica:**
Es el mejor indicador indirecto de fouling.
Más incrustación → canales más estrechos → mayor pérdida de carga → mayor presión diferencial


Esto se llama **fouling.**

**Valores realistas:**
Placas limpias: 0.2 – 0.4 bar
Placas sucias: 0.8 – 1.5 bar


9️⃣ valor_pu_microbiologico: Unidades de pasteurización (PU). Miden la letalidad térmica acumulada.

Parámetros
Tref = 72°C
z = 7°C

**Qué indica:**
Es el mejor indicador indirecto de fouling.
Más incrustación → canales más estrechos → mayor pérdida de carga → mayor presión diferencial


Esto se llama **fouling.**

**Valores realistas:**
Seguro: 15 – 40 PU
Excesivo: >100 PU


🔟 indice_seguridad: Variable binaria que indica si la pasteurización es segura.

Legalmente: PU ≥ 15

**Un dataset realista debería cumplir:**

temp_proceso → PU (muy fuerte)

flujo ↑ → tiempo ↓ → PU ↓

horas_desde_limpieza ↑ → fouling ↑

fouling ↑ → presión ↑

fouling ↑ → consumo_agua ↑

In [ ]:
import numpy as np
import pandas as pd
import os 

def generar_datos(n_samples=5000):

    np.random.seed(42)

    # ----------------------------
    # 1. CONDICIONES AMBIENTALES
    # ----------------------------

    temp_ambiente = np.random.uniform(12, 32, n_samples)

    temp_entrada_leche = (
        4 + 0.05*(temp_ambiente-20)
        + np.random.normal(0,0.4,n_samples)
    )

    # ----------------------------
    # 2. VARIABLES DE PRODUCCIÓN
    # ----------------------------

    flujo_leche_lh = np.random.uniform(2500,5000,n_samples)

    horas_desde_limpieza = np.random.uniform(0,20,n_samples)

    # ----------------------------
    # 3. CONTROL DE TEMPERATURA
    # ----------------------------

    temp_setpoint_leche = np.random.uniform(72,78,n_samples)

    temp_proceso_leche = (
        temp_setpoint_leche
        + np.random.normal(0,0.25,n_samples)
    )

    delta_t_servicio = np.random.uniform(6,14,n_samples)

    temp_agua_servicio = temp_proceso_leche + delta_t_servicio


    # ----------------------------
    # 4. FOULING
    # ----------------------------

    fouling = 1 - np.exp(-horas_desde_limpieza/6)

    # mayor temperatura acelera fouling
    fouling *= np.exp((temp_proceso_leche-72)/20)


    # ----------------------------
    # 5. PRESIÓN DIFERENCIAL
    # ----------------------------

    presion_diferencial_bar = (
        0.25
        + 0.8*fouling
        + flujo_leche_lh/25000
        + np.random.normal(0,0.03,n_samples)
    )


    # ----------------------------
    # 6. BALANCE TÉRMICO
    # ----------------------------

    cp_leche = 3.9

    calor_necesario = (
        flujo_leche_lh
        * cp_leche
        * (temp_proceso_leche-temp_entrada_leche)
    )


    eficiencia_termica = 0.9/(1+fouling*0.3)



    # ----------------------------
    # 7. CONSUMO DE AGUA
    # ----------------------------

    delta_t_agua = 15

    cp_agua = 4.18

    consumo_agua_enfriamiento = (calor_necesario/ (cp_agua*delta_t_agua))

    consumo_agua_enfriamiento /= eficiencia_termica

    consumo_agua_limpieza = 300 + 900*fouling

    consumo_agua_total = (
        consumo_agua_enfriamiento
        + consumo_agua_limpieza
    )

    consumo_agua_total *= np.random.uniform(0.98,1.02,n_samples)


    # ----------------------------
    # 8. SEGURIDAD MICROBIOLÓGICA
    # ----------------------------

    volumen_retencion = 16

    tiempo_residencia = (
        volumen_retencion/flujo_leche_lh
    )*3600

    Z = 7
    Tref = 72

    pu = (
        tiempo_residencia
        * 10**((temp_proceso_leche-Tref)/Z)
    )

    indice_seguridad = (pu>=15).astype(int)


    # ----------------------------
    # DATAFRAME FINAL
    # ----------------------------

    return pd.DataFrame({

        'temp_entrada_leche': temp_entrada_leche,
        'temp_ambiente': temp_ambiente,
        'temp_setpoint_leche': temp_setpoint_leche,
        'temp_proceso_leche': temp_proceso_leche,
        'temp_agua_servicio': temp_agua_servicio,
        'flujo_leche_lh': flujo_leche_lh,
        'horas_desde_limpieza': horas_desde_limpieza,
        'presion_diferencial_bar': presion_diferencial_bar,
        'consumo_agua_l': consumo_agua_total,
        'valor_pu_microbiologico': pu,
        'indice_seguridad': indice_seguridad
    })

In [2]:
# Generar y revisar
df_datagia = generar_datos(5000)
print(df_datagia.head())

   temp_entrada_leche  temp_ambiente  temp_setpoint_leche  temp_proceso_leche  \
0            3.735750      19.490802            74.151821           73.861484   
1            3.594593      31.014286            77.791366           77.549046   
2            4.167106      26.639879            76.187758           76.129340   
3            4.564048      23.973170            75.185138           75.454183   
4            3.971071      15.120373            77.813239           78.078340   

   temp_agua_servicio  flujo_leche_lh  horas_desde_limpieza  \
0           81.786188     3079.752804             19.175692   
1           90.838452     2616.744545             15.465223   
2           84.438972     4862.971449             18.119758   
3           88.005067     3284.338147              4.690617   
4           86.365838     3606.472558              3.951619   

   presion_diferencial_bar  consumo_vapor_kg  consumo_agua_l  \
0                 1.267604        548.648109    21014.466154   
1     

In [ ]:
os.makedirs('../../data/processed', exist_ok=True)
df_datagia.to_csv('../../data/processed/milk_dataset.csv', index=False)
print(f"\n✅ Dataset guardado en: data/processed/milk_dataset.csv")


✅ Dataset guardado en: data/processed/milk_dataset_v3.csv
